In [1]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

In [8]:
PROJECT_ROOT = "/Users/hlecates/Desktop/aqua-analytics"
CSV_PATH = f"{PROJECT_ROOT}/nescac/data/school-specific/fastest_times_by_school.csv"

OUT_GRID_DIR = f"{PROJECT_ROOT}/nescac/output/plots/schools/event-grids"
OUT_FASTEST_DIR = f"{PROJECT_ROOT}/nescac/output/plots/schools/event-fastest-counts"
OUT_INDIV_SCHOOL_DIR = f"{PROJECT_ROOT}/nescac/output/plots/schools/individual-event"
os.makedirs(OUT_INDIV_SCHOOL_DIR, exist_ok=True)
os.makedirs(OUT_FASTEST_DIR, exist_ok=True)
os.makedirs(OUT_GRID_DIR, exist_ok=True)

SCHOOLS = [
    "Amherst","Bates","Bowdoin","Colby","Connecticut College",
    "Hamilton","Middlebury","Trinity","Tufts","Wesleyan","Williams"
]

# consistent color order per school
import seaborn as sns
palette = dict(zip(SCHOOLS, sns.color_palette("tab20", len(SCHOOLS))))

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.7
})


In [3]:

df = pd.read_csv(CSV_PATH)

df = df[df["school"].isin(SCHOOLS)].copy()
df["year"] = df["year"].astype(int)

In [4]:
def sec_to_time(x, pos=None):
    if pd.isna(x):
        return ""
    x = float(x)
    m = int(x // 60)
    s = x - m * 60
    return f"{m}:{s:05.2f}" if m > 0 else f"{s:.2f}"

In [5]:
def apply_axis_style(ax, min_year, max_year, years_step=4):
    ax.yaxis.set_major_formatter(FuncFormatter(sec_to_time))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4, prune="both"))
    ax.tick_params(axis="y", labelsize=8)

    ax.set_xlim(min_year, max_year)
    if years_step:
        ax.set_xticks(list(range(min_year, max_year + 1, years_step)))
    ax.tick_params(axis="x", rotation=45, labelsize=8)

In [6]:
def safe_name(name: str) -> str:
    return (
        str(name)
        .replace("/", "-")
        .replace("\\", "-")
        .replace(" ", "_")
    )

In [ ]:
events = sorted(df["event_name"].unique())

for event in events:
    sub_event = df[df["event_name"] == event].copy()
    if sub_event.empty:
        continue

    min_year = int(sub_event["year"].min()) - 1
    max_year = int(sub_event["year"].max()) + 1

    # Grid: 3x4 = 12 axes; first 11 are line plots for schools, last is bar chart
    nrows, ncols = 3, 4
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 4, nrows * 3),
        sharex=False,  
        sharey=False
    )
    axes_flat = axes.flatten()
    line_axes = axes_flat[:11]
    bar_ax = axes_flat[-1]

    # Plot each school in its own panel
    for i, school in enumerate(SCHOOLS):
        ax = line_axes[i]
        s = sub_event[sub_event["school"] == school].sort_values("year")
        if not s.empty:
            ax.plot(
                s["year"], s["fastest_time_sec"],
                marker="o", linestyle="-", linewidth=1, markersize=3,
                color=palette.get(school, "#333")
            )
        apply_axis_style(ax, min_year, max_year, years_step=4)
        ax.set_title(school, fontsize=12, pad=5)

    # Unify y-limits across the 11 line plots 
    ymins, ymaxs = [], []
    for school in SCHOOLS:
        s = sub_event[sub_event["school"] == school]["fastest_time_sec"].dropna()
        if not s.empty:
            ymins.append(s.min())
            ymaxs.append(s.max())
    if ymins:
        y0, y1 = min(ymins), max(ymaxs)
        margin = 0.02 * max(1e-6, (y1 - y0))
        for ax in line_axes:
            ax.set_ylim(y0 - margin, y1 + margin)

    # Compute fastest-time counts per school for this event
    idx = sub_event.groupby("year")["fastest_time_sec"].idxmin()
    fastest_counts = (
        sub_event.loc[idx, "school"]
        .value_counts()
        .reindex(SCHOOLS, fill_value=0)
    )

    # 12th panel: bar chart
    bar_ax.clear()
    fc_nz = fastest_counts[fastest_counts > 0]
    if fc_nz.empty:
        bar_ax.text(0.5, 0.5, "No fastest-time counts", ha="center", va="center", fontsize=10, alpha=0.8)
        bar_ax.set_axis_off()
    else:
        fc_plot = fc_nz.sort_values(ascending=True)
        colors = [palette[s] for s in fc_plot.index]
        bar_ax.barh(fc_plot.index, fc_plot.values, color=colors)
        bar_ax.set_title("Fastest Time Counts", fontsize=12, pad=5)
        bar_ax.set_xlabel("Count (years with fastest time)")
        bar_ax.set_ylabel("")
        bar_ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        bar_ax.tick_params(axis="y", labelsize=8)
        bar_ax.grid(True, axis="x", linestyle="--", linewidth=0.5, alpha=0.7)
        bar_ax.invert_yaxis()
        xmax = int(fc_plot.values.max()) if len(fc_plot.values) else 0
        bar_ax.set_xlim(0, max(1, xmax + 1))

    # Save full fastest-time counts chart (includes zero-count schools)
    figf, axf = plt.subplots(figsize=(7, 6))
    fc_full = fastest_counts.sort_values(ascending=True)
    colors_full = [palette[s] for s in fc_full.index]
    axf.barh(fc_full.index, fc_full.values, color=colors_full)
    axf.set_title(f"{event.replace('_', ' ')} — Fastest Time Counts by School", fontsize=14, pad=8)
    axf.set_xlabel("Count (years with fastest time)")
    axf.set_ylabel("")
    axf.xaxis.set_major_locator(MaxNLocator(integer=True))
    axf.grid(True, axis="x", linestyle="--", linewidth=0.5, alpha=0.7)
    axf.tick_params(axis="y", labelsize=9)
    axf.invert_yaxis()
    plt.tight_layout()
    fastest_path = Path(OUT_FASTEST_DIR) / f"{safe_name(event)}_fastest_counts.png"
    figf.savefig(fastest_path, dpi=300, bbox_inches="tight")
    plt.close(figf)

    # Title and save the grid
    fig.suptitle(event.replace("_", " "), fontsize=16, fontweight="bold", y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    out_path = Path(OUT_GRID_DIR) / f"{safe_name(event)}.png"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

len(events)

15

In [9]:
events = sorted(df["event_name"].unique())

for school in SCHOOLS:
    school_dir = Path(OUT_INDIV_SCHOOL_DIR) / safe_name(school)
    school_dir.mkdir(parents=True, exist_ok=True)

    for event in events:
        sub = df[(df["school"] == school) & (df["event_name"] == event)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("year")
        min_year = int(sub["year"].min()) - 1
        max_year = int(sub["year"].max()) + 1

        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.plot(
            sub["year"], sub["fastest_time_sec"],
            marker="o", linestyle="-", linewidth=1.5, markersize=4,
            color=palette.get(school, "#333")
        )

        apply_axis_style(ax, min_year, max_year, years_step=4)
        ax.set_title(f"{school} — {event.replace('_', ' ')}", fontsize=14, pad=6)
        ax.set_xlabel("Year")
        ax.set_ylabel("Time (seconds)")

        plt.tight_layout()
        out_path = school_dir / f"{safe_name(event)}.png"
        fig.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close(fig)